## Sorting

Why does this matter?

Sorting is one of the most basic yet most used operations in data analysis. Before you present data to a manager, before you find top

performers, before you identify the worst-selling products — you sort. It makes patterns visible instantly.

The core methods —**sort_values() and sort_index()**

### What are they?

Two separate methods — **sort_values()** sorts by the actual data inside columns.

 **sort_index()** sorts by the index labels of the DataFrame.

### Why are there two methods?

Because a DataFrame has two things that can be sorted — the **index (row labels)** and the **values (actual data).**

 They serve different purposes. In 90% of real work you'll use **sort_values()**. **sort_index()** is used after operations that scramble your 
 
 index like **groupby**, **concat**, or **reset_index**.

### How does sorting work internally?

Pandas uses an optimized sorting algorithm (mergesort or quicksort depending on your choice) under the hood.

 It reorders the rows of the DataFrame based on the column values you specify,
 
  then returns a new DataFrame with those rows in the new order. The original index labels travel with the rows — they don't reset 
  
  automatically.

In [1]:
import pandas as pd
import numpy as np

data = {
    'Name':   ['Sumed', 'Priya', 'Rahul', 'Neha', 'Arjun'],
    'Dept':   ['Data',  'HR',    'Data',  'Finance', 'Data'],
    'Salary': [60000,  45000,  75000,  55000,    80000],
    'Exp':    [3,       2,       5,       4,         6]
}
df = pd.DataFrame(data)
print(df)


    Name     Dept  Salary  Exp
0  Sumed     Data   60000    3
1  Priya       HR   45000    2
2  Rahul     Data   75000    5
3   Neha  Finance   55000    4
4  Arjun     Data   80000    6


## Sort values(): sort by a single column


In [2]:
# sort_values(by='column') — sorts rows by that column's values
# ascending=True is DEFAULT → lowest to highest
# ascending=False → highest to lowest

# Sort by Salary — lowest first (default)
df.sort_values(by='Salary')

# Sort by Salary — highest first
df.sort_values(by='Salary', ascending=False)

# Sort by Name — alphabetical (strings sort A→Z by default)
df.sort_values(by='Name')

# WHY does index look scrambled after sorting?
# Because rows move but their original index labels travel with them
# Index 1 (Priya, lowest salary) is now first row
# Use reset_index(drop=True) if you want clean 0,1,2,3 after sort

,Name,Dept,Salary,Exp
4,Arjun,Data,80000,6
3,Neha,Finance,55000,4
1,Priya,HR,45000,2
2,Rahul,Data,75000,5
0,Sumed,Data,60000,3


*ℹ Index labels 4,2,0,3,1 — rows moved, original labels stayed attached. This is expected behavior.*

## Sort by multiple column


In [3]:
# Real world: sort by Dept first, then by Salary within each Dept
# Pass a LIST to 'by' for multi-column sort
# ascending can be a single bool (applies to all) OR a list of bools

df.sort_values(by=['Dept', 'Salary'])
# Both ascending (default)

df.sort_values(by=['Dept', 'Salary'], ascending=[True, False])
# Dept A→Z, but Salary highest first within each Dept

# HOW it works:
# First sorts entire df by Dept (primary key)
# Then within each Dept group, sorts by Salary (secondary key)
# The ascending list maps 1:1 to the by list

,Name,Dept,Salary,Exp
4,Arjun,Data,80000,6
2,Rahul,Data,75000,5
0,Sumed,Data,60000,3
3,Neha,Finance,55000,4
1,Priya,HR,45000,2


## NA_Position: controlling where NAN goes

In [5]:
# When a column has NaN values, where should they appear after sort?
# na_position='last'  → NaN goes to BOTTOM (default)
# na_position='first' → NaN goes to TOP

df_nan = df.copy()
df_nan.loc[2, 'Salary'] = np.nan   # introduce a NaN

df_nan.sort_values(by='Salary', na_position='last')
# NaN row appears at the bottom

df_nan.sort_values(by='Salary', na_position='first')
# NaN row appears at the top

# WHY does this matter?
# In reports and dashboards, you control whether incomplete
# records show at top (attention) or bottom (out of the way)

,Name,Dept,Salary,Exp
2,Rahul,Data,NaN,5
1,Priya,HR,45000.0,2
3,Neha,Finance,55000.0,4
0,Sumed,Data,60000.0,3
4,Arjun,Data,80000.0,6


## Sort Index(): sort by index values

In [6]:
# sort_index() sorts rows by their INDEX LABELS
# WHEN to use it:
# After groupby, merge, or concat — index gets scrambled
# sort_index() puts it back in order

# Simulate scrambled index (what happens after many operations)
df_scrambled = df.sort_values(by='Salary')
print(df_scrambled.index.tolist())   # [1, 3, 0, 2, 4]

# Restore original index order
df_scrambled.sort_index()
# ascending=False also works here
df_scrambled.sort_index(ascending=False)   # 4,3,2,1,0

[1, 3, 0, 2, 4]


,Name,Dept,Salary,Exp
4,Arjun,Data,80000,6
3,Neha,Finance,55000,4
2,Rahul,Data,75000,5
1,Priya,HR,45000,2
0,Sumed,Data,60000,3


## Nlargest() and Nsmallest(): Top N and bottom M

In [8]:
# nlargest(n, column) → top N rows by that column
# nsmallest(n, column) → bottom N rows by that column
# These are SHORTCUTS — faster and cleaner than sort + head

# Top 3 highest paid employees
df.nlargest(3, 'Salary')


,Name,Dept,Salary,Exp
4,Arjun,Data,80000,6
2,Rahul,Data,75000,5
0,Sumed,Data,60000,3


In [10]:

# Bottom 2 least experienced employees
df.nsmallest(2, 'Exp')

# WHY use nlargest instead of sort_values + head?
# sort_values sorts ALL rows then head() picks top N
# nlargest uses a heap algorithm — only finds top N
# Much faster on large datasets (millions of rows)



,Name,Dept,Salary,Exp
1,Priya,HR,45000,2
0,Sumed,Data,60000,3


In [11]:
# Equivalent but slower version:
df.sort_values('Salary', ascending=False).head(3)  # same result

,Name,Dept,Salary,Exp
4,Arjun,Data,80000,6
2,Rahul,Data,75000,5
0,Sumed,Data,60000,3


*In interviews and real dashboards, "top 10 customers by revenue" or "bottom 5 products by sales" — always reach for nlargest / nsmallest. It signals you know efficient Pandas.*

## KIND : choosing sort Algorithm

In [12]:
# sort_values has a 'kind' parameter for algorithm choice
# kind='quicksort'  → default, fastest on average
# kind='mergesort'  → stable sort (preserves original order of ties)
# kind='heapsort'   → memory efficient

# WHEN does 'kind' matter in real work?
# When you have TIES and order of tied rows matters
# Example: two employees with same salary
# mergesort keeps them in their original relative order
# quicksort may swap them — unpredictable for ties

df.sort_values(by='Salary', kind='mergesort')

# In 99% of DA work you don't need to specify kind
# But knowing it exists — and WHY — is an interview differentiator

,Name,Dept,Salary,Exp
1,Priya,HR,45000,2
3,Neha,Finance,55000,4
0,Sumed,Data,60000,3
2,Rahul,Data,75000,5
4,Arjun,Data,80000,6


*Stable sort = if two rows have the same value, their relative order from the original DataFrame is preserved. mergesort and stable (alias) are both stable. quicksort is not guaranteed to be stable.*

 ## Everything You Must Know Cold

**sort_values(by='col')** — sorts rows by data values. Default is ascending. Does NOT reset index — original labels travel with rows.

**ascending=[True, False]**— when sorting by multiple columns, pass a list of booleans matching the by list one-to-one.

**na_position** — controls where NaN lands after sort. *'last'* is default. *'first'* puts missing rows at top.

**sort_index()** — sorts by index labels, not values. Use after operations that scramble the index like groupby, merge, concat.

**nlargest(n, col) / nsmallest(n, col)** — faster than *sort_values + head()* for getting top/bottom N. Uses heap algorithm internally. 

Preferred in professional code.

**kind='mergesort'** — stable sort preserves relative order of tied rows. *quicksort* is default and fastest but unstable. Knowing this 

separates you in interviews.